In [ ]:
# run_pups_full_A_v2 — v2标签: C1_no_reloc 与 Mislocalized=0 合并为"不重定位"
import os, re, pickle, numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from pups_inference import load_model, esm2_encode, DEVICE

BASE = "/mnt/volume6/czj/labLGN/LabLZ/data_preparation/"
SRC  = BASE + "cell2024_model_single_subst.csv"
LM   = "/mnt/volume6/czj/labLGN/LabLZ/pups_trial/real_landmark.npy"
OUT  = "/mnt/volume6/czj/labLGN/LabLZ/pups_trial/pups_full_delta.pkl"
SAVE_EVERY = 20

def parse_pos(m):
    mt = re.match(r"^[A-Z](\d+)[A-Z]$", str(m).strip())
    return int(mt.group(1)) if mt else None

@torch.no_grad()
def probs(model, seq, lm_t, pos):
    X, xlen = esm2_encode(seq, pos=pos)
    _, multi = model.call_model(X, xlen, lm_t)
    return torch.sigmoid(multi).squeeze(0).cpu().numpy()

def main():
    df = pd.read_csv(SRC)
    df["KEY"] = df["Gene"].astype(str) + "||" + df["Variant"].astype(str)

    # ===== v2 标签: Mislocalized 为基础, C1_no_reloc 强制归 0 =====
    df["reloc_v2"] = df["Mislocalized"].astype(int)
    df.loc[df["label_5class"] == "C1_no_reloc", "reloc_v2"] = 0
    print(f"v2标签: reloc=0 {(df['reloc_v2']==0).sum()}例, "
          f"reloc=1 {(df['reloc_v2']==1).sum()}例")
    print(f"  Mislocalized=0:       {(df['Mislocalized']==0).sum()}")
    print(f"  C1_no_reloc→0:        {(df['label_5class']=='C1_no_reloc').sum()}")
    print(f"  C2-C5 (reloc=1):      {((df['reloc_v2']==1) & df['label_5class'].notna()).sum()}")

    lm_t = torch.from_numpy(np.load(LM).astype(np.float32)).unsqueeze(0).to(DEVICE)
    model = load_model()

    store = pickle.load(open(OUT, "rb")) if os.path.exists(OUT) else {}
    todo = df[~df["KEY"].isin(store)]
    print(f"待处理 {len(todo)} / 全量 {len(df)}")
    for i, (_, r) in enumerate(todo.iterrows()):
        wt, mt, pos = r.get("sequence"), r.get("mutant_sequence"), parse_pos(r.get("Mutation_used"))
        if not (isinstance(wt, str) and isinstance(mt, str) and pos):
            store[r["KEY"]] = None
        else:
            store[r["KEY"]] = (probs(model, mt, lm_t, pos) - probs(model, wt, lm_t, pos)).astype(np.float32)
        if (i + 1) % SAVE_EVERY == 0 or (i + 1) == len(todo):
            with open(OUT + ".tmp", "wb") as f:
                pickle.dump(store, f)
            os.replace(OUT + ".tmp", OUT)
            print(f"  Saved {i+1}/{len(todo)}")
    print(f"完成 {sum(v is not None for v in store.values())}/{len(store)} → {OUT}")

    # ===== v2 AUROC: 全量2179样本, L1(|Δ|) vs reloc_v2 =====
    df["l1"] = df["KEY"].map({k: float(np.abs(v).sum()) for k, v in store.items() if v is not None})
    valid = df.dropna(subset=["l1"])
    auroc = roc_auc_score(valid["reloc_v2"], valid["l1"])
    print(f"\nv2 AUROC(|Δ|→reloc_v2): {auroc:.3f}  (n={len(valid)})")

    # 保存 v2 标签 + l1 供后续使用
    valid[["KEY", "Gene", "reloc_v2", "l1"]].to_csv(BASE + "pups_l1_v2.csv", index=False)
    print(f"v2标签+l1 已存 → {BASE}pups_l1_v2.csv")

if __name__ == "__main__":
    main()

/mnt/volume6/czj/PUPS


I0000 00:00:1784032696.831153  737052 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784032697.901851  737052 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784032701.162235  737052 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


v2标签: reloc=0 1956例, reloc=1 223例
  Mislocalized=0:       1943
  C1_no_reloc→0:        13
  C2-C5 (reloc=1):      222
待处理 0 / 全量 2179
完成 2179/2179 → /mnt/volume6/czj/PUPS/LABCODE/pups_full_delta.pkl

v2 AUROC(|Δ|→reloc_v2): 0.582  (n=2179)
v2标签+l1 已存 → /mnt/volume6/czj/labLGN/LabLZ/pups_l1_v2.csv
